# 02b - Regresi Polinomial
Materi Modul 2: regresi polinomial, underfitting vs overfitting.

Regresi polinomial = regresi linier pada fitur yang dipangkatkan ($x, x^2, x^3, \dots$).
Model tetap **linier terhadap koefisien**, tapi bisa membentuk kurva.

Dataset: `kurva_sinus.csv` (data sinus + noise, 60 titik) supaya overfitting terlihat jelas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

data = pd.read_csv('../datasets/kurva_sinus.csv')
X = data[['x']]
y = data['y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

plt.scatter(X_train, y_train, label='Train')
plt.scatter(X_test, y_test, color='orange', label='Test')
plt.legend(); plt.title('Data kurva sinus'); plt.show()

## 1. Fitur polinomial

In [ ]:
pf = PolynomialFeatures(degree=3, include_bias=False)
X_poly = pf.fit_transform(X_train[:3])
print(pf.get_feature_names_out())
print(X_poly.round(4))

## 2. Bandingkan derajat polinomial

In [ ]:
x_plot = np.linspace(0, 1, 200).reshape(-1, 1)
x_plot_df = pd.DataFrame(x_plot, columns=['x'])

fig, ax = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for a, derajat in zip(ax, [1, 3, 9, 15]):
    model = make_pipeline(PolynomialFeatures(derajat), LinearRegression())
    model.fit(X_train, y_train)
    a.scatter(X_train, y_train, s=15)
    a.plot(x_plot, model.predict(x_plot_df), color='red')
    a.set_ylim(-2, 2)
    a.set_title(f'Derajat {derajat}')
plt.tight_layout(); plt.show()

## 3. Error train vs test per derajat
Train error terus turun, tapi test error naik lagi setelah titik tertentu = **overfitting**.

In [ ]:
derajat_list = range(1, 16)
err_train, err_test = [], []

for derajat in derajat_list:
    model = make_pipeline(PolynomialFeatures(derajat), LinearRegression())
    model.fit(X_train, y_train)
    err_train.append(mean_squared_error(y_train, model.predict(X_train)))
    err_test.append(mean_squared_error(y_test, model.predict(X_test)))

plt.plot(derajat_list, err_train, marker='o', label='Train MSE')
plt.plot(derajat_list, err_test, marker='o', label='Test MSE')
plt.yscale('log')
plt.xlabel('Derajat polinomial'); plt.ylabel('MSE'); plt.legend()
plt.title('Underfitting -> pas -> overfitting')
plt.show()

terbaik = list(derajat_list)[int(np.argmin(err_test))]
print('Derajat dengan test MSE terkecil:', terbaik)

## 4. Regresi polinomial pada data nyata (Diabetes)

In [ ]:
df = pd.read_csv('../datasets/diabetes.csv')
Xd = df.drop(columns='progres_penyakit')
yd = df['progres_penyakit']
Xd_train, Xd_test, yd_train, yd_test = train_test_split(Xd, yd, test_size=0.2, random_state=42)

for derajat in [1, 2, 3]:
    model = make_pipeline(PolynomialFeatures(derajat, include_bias=False), LinearRegression())
    model.fit(Xd_train, yd_train)
    print(f'Derajat {derajat} | jumlah fitur={model[0].n_output_features_:4d} | R2 train={model.score(Xd_train, yd_train):.3f} | R2 test={model.score(Xd_test, yd_test):.3f}')

## Catatan
- (apa itu underfitting/overfitting, bagaimana memilih derajat, kenapa fitur polinomial menambah jumlah fitur sangat cepat, dll.)